In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Conv1D, GlobalMaxPooling1D, Concatenate,BatchNormalization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import time
from tensorflow.keras.layers import MaxPooling1D
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
train = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_train_final.csv')
test = pd.read_csv('/content/drive/My Drive/HTC_CNN_dataset/WOS/wos_test_final.csv')

In [ ]:
data = train

texts = data['text'].values
labels_l1 = data['l1'].values
labels_l2 = data['l2'].values

label_encoder_l1 = LabelEncoder()
label_encoder_l2 = LabelEncoder()

encoded_l1 = label_encoder_l1.fit_transform(labels_l1)
encoded_l2 = label_encoder_l2.fit_transform(labels_l2)

tokenizer = Tokenizer(num_words=4000)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X_padded = pad_sequences(sequences, maxlen=400)

y_l1 = to_categorical(encoded_l1)
y_l2 = to_categorical(encoded_l2)

In [ ]:
data_test = test

texts_test = data_test['text'].values
labels_l1_test = data_test['l1'].values
labels_l2_test = data_test['l2'].values

encoded_l1_test = label_encoder_l1.transform(labels_l1_test)
encoded_l2_test = label_encoder_l2.transform(labels_l2_test)

sequences_test = tokenizer.texts_to_sequences(texts_test)
X_padded_test = pad_sequences(sequences_test, maxlen=400)

y_l1_test = to_categorical(encoded_l1_test)
y_l2_test = to_categorical(encoded_l2_test)

In [ ]:
input_text = Input(shape=(400,))

embedding_text = Embedding(input_dim=4000, output_dim=64, input_length=400)(input_text)

conv_text = Conv1D(64, 3, activation='relu')(embedding_text)

global_pool_text = GlobalMaxPooling1D()(conv_text)

dense_text_l1 = Dense(64, activation='relu')(global_pool_text)

output_l1 = Dense(len(label_encoder_l1.classes_), activation='softmax')(dense_text_l1)

model_l1 = Model(inputs=input_text, outputs=output_l1)
model_l1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
begin_l1_time = time.time()
history_l1 = model_l1.fit(X_padded, y_l1, epochs=10, batch_size=128, validation_split=0.2, callbacks=[early_stopping])
end_l1_time = time.time()
l1_time = round(end_l1_time - begin_l1_time)
print(f'Level 1 model training time: {l1_time:.2f} seconds')

Epoch 1/10
196/196 [==============================] - 27s 130ms/step - loss: 1.2548 - accuracy: 0.5343 - val_loss: 0.6474 - val_accuracy: 0.7884
Epoch 2/10
196/196 [==============================] - 6s 32ms/step - loss: 0.3905 - accuracy: 0.8667 - val_loss: 0.3699 - val_accuracy: 0.8759
Epoch 3/10
196/196 [==============================] - 6s 32ms/step - loss: 0.2449 - accuracy: 0.9165 - val_loss: 0.3415 - val_accuracy: 0.8857
Epoch 4/10
196/196 [==============================] - 4s 18ms/step - loss: 0.1646 - accuracy: 0.9488 - val_loss: 0.3454 - val_accuracy: 0.8874
Epoch 5/10
196/196 [==============================] - 3s 18ms/step - loss: 0.1020 - accuracy: 0.9743 - val_loss: 0.3635 - val_accuracy: 0.8847
l1 模型训练时间: 47.00 秒


In [ ]:
pred_l1 = model_l1.predict(X_padded)
pred_labels_l1 = label_encoder_l1.inverse_transform(np.argmax(pred_l1, axis=1))
accuracy_l1 = np.mean(pred_labels_l1 == labels_l1)
print(f"Training accuracy for l1: {accuracy_l1:.4f}")


pred_l1_test = model_l1.predict(X_padded_test)
pred_labels_l1_test = label_encoder_l1.inverse_transform(np.argmax(pred_l1_test, axis=1))
accuracy_l1_test = np.mean(pred_labels_l1_test == labels_l1_test)

print(f"Testing accuracy for l1: {accuracy_l1_test:.4f}")

979/979 [==============================] - 2s 2ms/step
Training accuracy for l1: 0.9415
490/490 [==============================] - 1s 3ms/step
Testing accuracy for l1: 0.8849


In [ ]:
input_l1 = Input(shape=(len(label_encoder_l1.classes_),))
input_text = Input(shape=(400,))
embedding_text = Embedding(input_dim=4000, output_dim=64, input_length=400)(input_text)

conv_text = Conv1D(filters=64, kernel_size=3, activation='relu')(embedding_text)
# pool_text = MaxPooling1D(pool_size=3)(conv_text)

global_pool_text = GlobalMaxPooling1D()(conv_text)
dense_text = Dense(64, activation='relu')(global_pool_text)
concatenated_l2 = Concatenate()([dense_text, input_l1])
dense_l2 = Dense(len(label_encoder_l2.classes_), activation='relu')(concatenated_l2)
output_l2 = Dense(len(label_encoder_l2.classes_), activation='softmax')(dense_l2)

model_l2 = Model(inputs=[input_text, input_l1], outputs=output_l2)
model_l2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
begin_l2_time = time.time()
history_l2 = model_l2.fit([X_padded, y_l1], y_l2, epochs=10, batch_size=128,  validation_split=0.2, callbacks=[early_stopping])
end_l2_time = time.time()
l2_time = round(end_l2_time - begin_l2_time)
print(f'Level 2 model training time: {l2_time:.2f} seconds')

Epoch 1/10
196/196 [==============================] - 15s 70ms/step - loss: 3.7452 - accuracy: 0.1501 - val_loss: 2.2328 - val_accuracy: 0.4350
Epoch 2/10
196/196 [==============================] - 6s 33ms/step - loss: 1.5145 - accuracy: 0.6279 - val_loss: 1.1157 - val_accuracy: 0.7118
Epoch 3/10
196/196 [==============================] - 5s 26ms/step - loss: 0.8840 - accuracy: 0.7761 - val_loss: 0.9268 - val_accuracy: 0.7557
Epoch 4/10
196/196 [==============================] - 4s 19ms/step - loss: 0.6413 - accuracy: 0.8323 - val_loss: 0.8547 - val_accuracy: 0.7740
Epoch 5/10
196/196 [==============================] - 3s 15ms/step - loss: 0.4725 - accuracy: 0.8747 - val_loss: 0.8632 - val_accuracy: 0.7787
Epoch 6/10
196/196 [==============================] - 2s 10ms/step - loss: 0.3431 - accuracy: 0.9109 - val_loss: 0.8733 - val_accuracy: 0.7835
l2 模型训练时间: 36.00 秒


In [ ]:
model_l2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_l2.save('model_l2_frozen.h5')

pred_l2= model_l2.predict([X_padded, y_l1])
pred_labels_l2 = label_encoder_l2.inverse_transform(np.argmax(pred_l2, axis=1))
accuracy_l2 = np.mean(pred_labels_l2 == labels_l2)
print(f'Level 2 Accuracy (Training Set): {accuracy_l2:.4f}')


X_padded_test_l2 = to_categorical(label_encoder_l1.transform(pred_labels_l1_test))
pred_l2_test = model_l2.predict([X_padded_test,X_padded_test_l2 ])
pred_labels_l2_test = label_encoder_l2.inverse_transform(np.argmax(pred_l2_test, axis=1))
accuracy_l2_test = np.mean(pred_labels_l2_test == labels_l2_test)
print(f'Level 2 Accuracy (Test Set): {accuracy_l2_test:.4f}')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


979/979 [==============================] - 2s 2ms/step
训练集l2准确率: 0.8603
490/490 [==============================] - 1s 3ms/step
测试集l2准确率: 0.7294


In [ ]:
correct_l1 = pred_labels_l1_test == labels_l1_test
accuracy_l1 = np.mean(correct_l1)
print(f"Consistency Accuracy for l1: {accuracy_l1:.4f}")

correct_l1_indices = np.where(correct_l1)[0]
pred_labels_l2_filtered = pred_labels_l2_test[correct_l1_indices]
labels_l2_filtered = labels_l2_test[correct_l1_indices]
correct_l2 = pred_labels_l2_filtered == labels_l2_filtered
accuracy_l2 = np.sum(correct_l2) / len(labels_l2_test)
print(f"Consistency Accuracy for l2: {accuracy_l2:.4f}")

Consistency Accuracy for l1: 0.8849
Consistency Accuracy for l2: 0.7263
